In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
import random
from PIL import Image
from torchvision import transforms

In [17]:
# ================= CONFIG =================
DATASET_ROOT = "dataset"
IMG_SIZE = 224
EPOCHS = 20

IMG_DIR = os.path.join(DATASET_ROOT, "train")
LABEL_DIR = os.path.join(DATASET_ROOT, "labels")

In [11]:
# ================= FILE MATCHING =================
images = []

for file in os.listdir(IMG_DIR):
    if file.endswith(".jpg") or file.endswith(".png"):
        label_file = file.replace(".jpg", ".txt").replace(".png", ".txt")
        if os.path.exists(os.path.join(LABEL_DIR, label_file)):
            images.append(file)

random.shuffle(images)

split_idx = int(0.8 * len(images))
train_files = images[:split_idx]
val_files = images[split_idx:]

In [12]:
# ================= TRANSFORMS =================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

In [13]:
# ================= DATASET =================
class YoloDataset(torch.utils.data.Dataset):
    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]

        img_path = os.path.join(IMG_DIR, img_name)
        label_path = os.path.join(LABEL_DIR, img_name.replace(".jpg",".txt").replace(".png",".txt"))

        # ===== IMAGE (PIL) =====
        img = Image.open(img_path).convert("RGB")
        img = transform(img)

        # ===== LABEL =====
        with open(label_path) as f:
            lines = f.readlines()

        line = lines[0].strip().split()
        cls, xc, yc, w, h = map(float, line)

        target = torch.tensor([xc, yc, w, h, 1.0], dtype=torch.float32)

        return img, target

In [14]:
# ================= MODEL =================
class SimpleYOLO(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3,16,3,1,1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,1,1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,1,1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.head = nn.Linear(64, 5)

    def forward(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        return self.head(x)

In [15]:
# ================= TRAIN =================
device = "cuda" if torch.cuda.is_available() else "cpu"

train_dataset = YoloDataset(train_files)
val_dataset = YoloDataset(val_files)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16)

model = SimpleYOLO().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [16]:
# ================= TRAIN LOOP =================
for epoch in range(EPOCHS):
    print("--- TRAINING STARTED ---")
    model.train()
    train_loss = 0

    for imgs, targets in train_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)

        preds = model(imgs)

        loss_bbox = ((preds[:,:4] - targets[:,:4])**2).mean()
        loss_obj = nn.BCEWithLogitsLoss()(preds[:,4], targets[:,4])

        loss = loss_bbox + loss_obj

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # ===== VALIDATION =====
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = imgs.to(device)
            targets = targets.to(device)

            preds = model(imgs)

            loss_bbox = ((preds[:,:4] - targets[:,:4])**2).mean()
            loss_obj = nn.BCEWithLogitsLoss()(preds[:,4], targets[:,4])

            val_loss += (loss_bbox + loss_obj).item()

    print(f"Epoch {epoch} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")
print("--- TRAINING ENDED ---")

Epoch 0 | Train: 7.8087 | Val: 1.1194
Epoch 1 | Train: 2.1458 | Val: 0.3256
Epoch 2 | Train: 0.9838 | Val: 0.2360
Epoch 3 | Train: 0.6739 | Val: 0.1929
Epoch 4 | Train: 0.5772 | Val: 0.2001
Epoch 5 | Train: 0.5475 | Val: 0.1844
Epoch 6 | Train: 0.5277 | Val: 0.1941
Epoch 7 | Train: 0.5439 | Val: 0.1916
Epoch 8 | Train: 0.5245 | Val: 0.1837
Epoch 9 | Train: 0.5173 | Val: 0.1804
Epoch 10 | Train: 0.4945 | Val: 0.1922
Epoch 11 | Train: 0.5126 | Val: 0.1956
Epoch 12 | Train: 0.5215 | Val: 0.2051
Epoch 13 | Train: 0.5031 | Val: 0.1847
Epoch 14 | Train: 0.4986 | Val: 0.1847
Epoch 15 | Train: 0.4989 | Val: 0.1834
Epoch 16 | Train: 0.4751 | Val: 0.1863
Epoch 17 | Train: 0.4795 | Val: 0.2273
Epoch 18 | Train: 0.5196 | Val: 0.2029
Epoch 19 | Train: 0.5024 | Val: 0.2044


In [18]:
# ================= SAVE =================
scripted = torch.jit.script(model)
scripted.save("ball_yolo.pth")